100 equilibria, use one free parameter combination (or maybe 5) and then run the EPEDNN loop and make a plot of pressure predicted vs. pfile pressure at psiN=1-delta and psiN=0.85

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np

ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))
from src.profiles_loop_solve import profiles_loop_solve

In [ ]:
verbose = False

# Output data and files
ne_success_fp = 'success_profloop'

# Scan parameters
x_res = 40
eped_tol_max = 1e-2
free_params = {
    'alpha_crit': 3.16,
    'C_KBM': 0.1,
    'De_chie_etg': 3.16,
    'nFC_x0': 1e16,
    'ncx_x0_ratio': 2.44
}

# Equilibria parameters
equil_num = 100
geqdsk_dir = Path('/mnt/homes_global/jal2351/software/sc_inputs/CAKEgeqdsks')
pfile_dir = Path('/mnt/homes_global/jal2351/software/sc_inputs/CAKEpfiles')


In [ ]:
# Load in equilibria
from collections import defaultdict

def initialize_inputs(equil_num, geqdsk_dir=geqdsk_dir, pfile_dir=pfile_dir):
    """Select equil_num g/p file pairs from the CAKE input directories.

    Files are named g{shot}.{time} and p{shot}.{time}. Selection prioritizes
    one equilibrium per shot number before adding additional times from shots
    that already have a selected equilibrium.
    """
    geqdsk_dir = Path(geqdsk_dir)
    pfile_dir = Path(pfile_dir)

    g_by_suffix = {
        f.name[1:]: f for f in geqdsk_dir.glob("g*") if f.is_file()
    }
    p_by_suffix = {
        f.name[1:]: f for f in pfile_dir.glob("p*") if f.is_file()
    }
    shared_suffixes = sorted(set(g_by_suffix) & set(p_by_suffix))

    by_shot = defaultdict(list)
    for suffix in shared_suffixes:
        shot, time = suffix.split(".", 1)
        by_shot[shot].append((time, suffix))
    for shot in by_shot:
        by_shot[shot].sort()

    shots = sorted(by_shot)
    selected = []
    time_idx = 0
    while len(selected) < equil_num:
        added_this_round = False
        for shot in shots:
            if len(selected) >= equil_num:
                break
            entries = by_shot[shot]
            if time_idx < len(entries):
                suffix = entries[time_idx][1]
                selected.append(
                    (str(g_by_suffix[suffix]), str(p_by_suffix[suffix]))
                )
                added_this_round = True
        if not added_this_round:
            break
        time_idx += 1

    if len(selected) < equil_num:
        raise ValueError(
            f"Requested {equil_num} equilibria but only found {len(selected)} "
            f"matching g/p pairs in {geqdsk_dir} and {pfile_dir}"
        )

    print(f"Selected {len(selected)} g/p file pairs:")
    for mhd_fp, kprof_fp in selected:
        print(f"  g: {mhd_fp}\n  p: {kprof_fp}")
    return selected

equilibria = initialize_inputs(equil_num, geqdsk_dir, pfile_dir)

In [ ]:
# For each equilbrium, call profiles_loop_solve() from profiles_loop_solve.py
failed=[]
for mhd_fp, kprof_fp in equilibria:
    try:
        profiles_loop_solve(
            MHD_FP = mhd_fp,
            KPROF_FP = kprof_fp,
            ne_success_fp = ne_success_fp,
            x_res = x_res,
            eped_tol_max = eped_tol_max,
            free_params = free_params,
            verbose = verbose,
        )
    except Exception as e:
        failed.append(mhd_fp)
    else:
        # need a way to save output of profiles_loop_solve(), aka final T and ne profile, for plotting in next cell

In [ ]:
# Plot pressure predicted vs. pfile pressure at psiN=1-delta and psiN=0.85 for all equilibria